# Clair — Quantization & Export to GGUF

This notebook merges the Clair LoRA adapters with the base model and exports to GGUF format for CPU inference with llama.cpp.

## Steps:
1. Merge LoRA adapters with base model
2. Export to multiple GGUF quantization formats
3. Test inference speed
4. Select best model for 7GB RAM constraint

## Target Quantizations:
- Q4_K_M: ~1.8GB (recommended for 7GB RAM)
- Q5_K_M: ~2.1GB (better quality, more RAM)
- Q3_K_M: ~1.5GB (smaller, lower quality)

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
import subprocess
import time

# Configuration
base_model_name = "Qwen/Qwen2.5-3B-Instruct"
lora_path = "models/clair-lora"
output_dir = "models/clair-gguf"
max_seq_length = 2048

os.makedirs(output_dir, exist_ok=True)

print(f"Base model: {base_model_name}")
print(f"LoRA adapters: {lora_path}")
print(f"Output directory: {output_dir}")

## 1. Load Model with LoRA Adapters

In [ ]:
# Load base model with LoRA adapters
print("Loading model with LoRA adapters...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_path,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("✓ Model loaded successfully")

## 2. Merge LoRA Adapters

In [ ]:
# Merge LoRA adapters into base model
merged_model_path = "models/merged/clair"
os.makedirs(merged_model_path, exist_ok=True)

print("Merging LoRA adapters with base model...")
model.save_pretrained_merged(merged_model_path, tokenizer, save_method="merged_16bit")

print(f"✓ Merged model saved to: {merged_model_path}")

## 3. Export to GGUF Format

In [ ]:
# Export to multiple quantization formats
quantization_methods = [
    ("q4_k_m", "Q4_K_M"),  # Recommended for 7GB RAM
    ("q5_k_m", "Q5_K_M"),  # Better quality
    ("q3_k_m", "Q3_K_M"),  # Smaller size
]

print("Exporting to GGUF formats...\n")

for method, name in quantization_methods:
    output_file = f"{output_dir}/clair-{name.lower()}.gguf"
    print(f"Exporting to {name}...")
    
    model.save_pretrained_gguf(
        output_file,
        tokenizer,
        quantization_method=method,
    )
    
    # Get file size
    if os.path.exists(output_file):
        size_gb = os.path.getsize(output_file) / (1024**3)
        print(f"✓ Saved: {output_file} ({size_gb:.2f} GB)\n")
    else:
        print(f"✗ Failed to export {name}\n")

## 4. Test Inference Speed

In [ ]:
# Install llama-cpp-python if not already installed
try:
    from llama_cpp import Llama
    print("✓ llama-cpp-python is installed")
except ImportError:
    print("Installing llama-cpp-python...")
    !pip install llama-cpp-python
    from llama_cpp import Llama
    print("✓ llama-cpp-python installed")

In [ ]:
# Test each quantization
test_prompt = "What is your name?"
results = []

for method, name in quantization_methods:
    model_file = f"{output_dir}/clair-{name.lower()}.gguf"
    
    if not os.path.exists(model_file):
        print(f"✗ {model_file} not found, skipping")
        continue
    
    print(f"\nTesting {name}...")
    
    # Load model
    llm = Llama(
        model_path=model_file,
        n_ctx=2048,
        n_threads=4,  # Simulate Intel i5 (4 cores)
        verbose=False,
    )
    
    # Warm-up
    _ = llm(test_prompt, max_tokens=10)
    
    # Benchmark
    start_time = time.time()
    output = llm(test_prompt, max_tokens=100)
    elapsed = time.time() - start_time
    
    # Calculate tokens/sec
    tokens_generated = len(output['choices'][0]['text'].split())
    tokens_per_sec = tokens_generated / elapsed
    
    results.append({
        'quantization': name,
        'size_gb': os.path.getsize(model_file) / (1024**3),
        'tokens_per_sec': tokens_per_sec,
        'time_sec': elapsed,
    })
    
    print(f"  Size: {results[-1]['size_gb']:.2f} GB")
    print(f"  Speed: {tokens_per_sec:.1f} tokens/sec")
    print(f"  Time: {elapsed:.2f} sec")
    print(f"  Response: {output['choices'][0]['text'][:100]}...")
    
    # Clean up
    del llm

## 5. Summary & Recommendations

In [ ]:
# Upload to OSS (optional)
print("\n📦 Upload to Alibaba Cloud OSS:")
print("ossutil cp -r models/clair-gguf/ oss://zim-my-models/clair-gguf/")
print("\n# Download to local laptop:")
print("ossutil cp -r oss://zim-my-models/clair-gguf/ ./models/clair-gguf/")

## 📊 Quantization Summary

### File Sizes
- `models/clair-gguf/clair-q4_k_m.gguf` (~1.8GB) - **Recommended**
- `models/clair-gguf/clair-q5_k_m.gguf` (~2.1GB) - Better quality
- `models/clair-gguf/clair-q3_k_m.gguf` (~1.5GB) - Smaller size

### Performance
The Q4_K_M quantization provides the best balance of:
- **Size**: ~1.8GB (leaves 5+ GB for KV cache + app)
- **Quality**: Minimal quality loss vs full precision
- **Speed**: Fast CPU inference on Intel i5

### Next Steps
1. Download GGUF files to local laptop
2. Test inference with Clair personality
3. Build demo app with Streamlit
4. Benchmark on target hardware (must stay < 7GB RAM)

In [ ]:
# Instructions for downloading to local machine
print("\n" + "="*60)
print("DOWNLOAD INSTRUCTIONS")
print("="*60)

print("""
To download the quantized Clair models to your local machine:

Option 1: Using ossutil (Alibaba Cloud OSS)
-------------------------------------------
# Upload to OSS
ossutil cp -r models/clair-gguf/ oss://zim-my-models/clair-gguf/

# Download to local laptop
ossutil cp -r oss://zim-my-models/clair-gguf/ ./models/clair-gguf/

Option 2: Using scp (direct download)
-------------------------------------
# From your local machine:
scp -r <pai-dsw-ip>:/mnt/workspace/zim-my/models/clair-gguf/ ./models/clair-gguf/

Option 3: Using PAI-DSW web interface
-------------------------------------
1. Open PAI-DSW file browser
2. Navigate to models/clair-gguf/
3. Download each .gguf file individually

Recommended file: clair-q4_k_m.gguf (~1.8GB)
""")

print("\n" + "="*60)
print("CLAIR QUANTIZATION COMPLETE")
print("="*60)
print("\nClair is ready for local deployment!")
print("\nClair knows:")
print("  • Its name is Clair")
print("  • Created by Michael Mlungisi Nkomo")
print("  • Made in Zimbabwe")
print("\n" + "="*60)